In [1]:
#loading filtered rna and atac data
import anndata
from anndata import AnnData
import pandas as pd

filtered_ATdata = anndata.read_h5ad("/home/fgsasse_lrs_1/Downloads/BA/BA_data/ATAC/zf_multiome_atlas_full_ATAC_v1_subset.h5ad")
filtered_ATdata.obs

#loading the columns of the rna data that should be added to the atac data
filtered_Rcol = pd.read_csv("/home/fgsasse_lrs_1/Downloads/BA/BA_data/ATAC/rna_fil.csv")
filtered_Rcol.head()

,cell_id,zebrafish_anatomy_ontology_class
0,AAACAGCCACCTAAGC-1_1,epidermis
1,AAACAGCCAGGGAGGA-1_1,pronephros
2,AAACAGCCATAGACCC-1_1,hindbrain
3,AAACATGCAAACTCAT-1_1,spinal_cord
4,AAACATGCAAGGACCA-1_1,neural_optic2


In [2]:
# before adding the column to the ATAC data, we need to make sure that the order of the cells in the ATAC data matches the order of the cells in the RNA data. We can do this by sorting both datasets by their cell IDs.

filtered_Rcol["cell_id"] == filtered_ATdata.obs.index


0        True
1        True
2        True
3        True
4        True
         ... 
93058    True
93059    True
93060    True
93061    True
93062    True
Length: 93063, dtype: bool

In [3]:
# Add RNA annotation to ATAC obs by matching on cell_id
rna_annotation = (
    filtered_Rcol.set_index("cell_id")["zebrafish_anatomy_ontology_class"]
    .reindex(filtered_ATdata.obs.index)
 )
filtered_ATdata.obs["annotation_ML"] = rna_annotation.values
filtered_ATdata.obs.head()

,orig.ident,nCount_RNA,nFeature_RNA,nCount_ATAC,nFeature_ATAC,nucleosome_signal,nucleosome_percentile,TSS.enrichment,TSS.percentile,nCount_SCT,...,leiden_4_merged,leiden_5_merged,leiden_6_merged,leiden_7_merged,leiden_8_merged,leiden_9_merged,leiden_10_merged,annotation_ML_coarse,dev_stage,annotation_ML
AAACAGCCACCTAAGC-1_1,SeuratProject,6522.0,2317,21425.0,9781,0.571984,0.40,4.488213,0.48,5661.0,...,76,84,92,91,92,93,94,epidermis,15somites,epidermis
AAACAGCCAGGGAGGA-1_1,SeuratProject,6100.0,2319,10334.0,5028,0.448143,0.13,4.795205,0.75,5553.0,...,22,15,7,68,61,56,57,pronephros,15somites,pronephros
AAACAGCCATAGACCC-1_1,SeuratProject,12581.0,3467,51485.0,19874,0.514213,0.24,5.238692,0.92,5781.0,...,77,88,96,104,100,102,104,hindbrain,15somites,hindbrain
AAACATGCAAACTCAT-1_1,SeuratProject,5642.0,2145,19812.0,9183,0.673319,0.85,4.409525,0.41,5363.0,...,3,1,102,109,107,110,114,spinal_cord,15somites,spinal_cord
AAACATGCAAGGACCA-1_1,SeuratProject,2691.0,838,5182.0,2565,0.394904,0.06,4.939061,0.83,4727.0,...,40,37,68,70,66,61,61,neural_optic,15somites,neural_optic2


## Pseudobulking the ATAC-Seq data
- creating new matrix of raw counts of ATAC-Seq data 
- creating df for metadata 
- aggregating the counts by 1) cell type, 2) cell type and developmental stages
    - aggregation equation: total count of a gene / total count of the matrix   
This normalizes each gene's count by the total counts within each cell type group, giving a proportional representation rather than raw counts.

In [4]:
# creating an aggregation function to sum counts for each cell type and normalize by total counts
def agg_fun(mat, meta, fun="sum", peaks=None):
    import pandas as pd
    import numpy as np

    meta = pd.Series(meta)

    if fun != "sum":
        raise ValueError("Only fun='sum' is supported.")

    def agg_f(mask):
        mask = np.asarray(mask)  # safe for sparse indexing
        sub = mat[mask, :]
        col_sums = np.asarray(sub.sum(axis=0)).ravel()  # 1D array
        total = col_sums.sum()
        if total == 0:
            return np.zeros_like(col_sums, dtype=float)
        return col_sums / total

    groups = meta.unique()
    agg_result = pd.DataFrame(
        [agg_f(meta == group) for group in groups],
        index=groups,
        columns= peaks if peaks is not None else None,
    )
    return agg_result

In [5]:
# Keep matrix sparse to avoid memory errors & extract metadata
atac_ct_matrix = filtered_ATdata.X
cell_types = filtered_ATdata.obs["annotation_ML"].astype(str)
dev_times = filtered_ATdata.obs["dev_stage"].astype(str)
peaks = filtered_ATdata.var_names
print(filtered_ATdata.obs)

                         orig.ident  nCount_RNA  nFeature_RNA  nCount_ATAC  \
AAACAGCCACCTAAGC-1_1  SeuratProject      6522.0          2317      21425.0   
AAACAGCCAGGGAGGA-1_1  SeuratProject      6100.0          2319      10334.0   
AAACAGCCATAGACCC-1_1  SeuratProject     12581.0          3467      51485.0   
AAACATGCAAACTCAT-1_1  SeuratProject      5642.0          2145      19812.0   
AAACATGCAAGGACCA-1_1  SeuratProject      2691.0           838       5182.0   
...                             ...         ...           ...          ...   
TTTGTGTTCCCTCAGT-1_7  SeuratProject      3079.0          1281      36101.0   
TTTGTTGGTACCTTAC-1_7  SeuratProject      6276.0          2441      71447.0   
TTTGTTGGTATTGAGT-1_7  SeuratProject      2377.0          1164      49177.0   
TTTGTTGGTGCGCGTA-1_7  SeuratProject      1750.0           811      12537.0   
TTTGTTGGTTAAGGCC-1_7  SeuratProject      8999.0          2399       3744.0   

                      nFeature_ATAC  nucleosome_signal  nucleos

In [6]:
#Aggregate by cell type
agg_atac_ct = agg_fun(atac_ct_matrix, cell_types, fun="sum", peaks=peaks)
agg_atac_ct.head()

,1-32-526,1-2372-3057,1-3427-4032,1-4469-7268,1-9541-9969,1-11007-12962,1-13276-13705,1-14059-14260,1-14625-15105,1-15724-15934,...,25-37489449-37490741,25-37492035-37493157,25-37493175-37493740,25-37496420-37496948,25-37497049-37497789,25-37498106-37500090,25-37500598-37500859,25-37501104-37501839,MT-22-3567,MT-13233-16532
epidermis,3.008003e-07,6.316807e-07,0.000002,0.000014,0.000003,0.000036,0.000003,0.000006,0.000024,0.000003,...,0.000026,0.000002,5.113605e-07,6.166406e-07,8.272009e-07,0.000004,1.955202e-07,7.520008e-07,0.002432,0.002169
pronephros,1.792938e-07,6.574107e-07,0.000001,0.000018,0.000004,0.000041,0.000003,0.000005,0.000014,0.000003,...,0.000027,0.000002,4.581954e-07,6.574107e-07,8.367046e-07,0.000004,2.988231e-07,9.761554e-07,0.002704,0.002405
hindbrain,2.542045e-07,9.623457e-07,0.000002,0.000015,0.000003,0.000036,0.000002,0.000004,0.000013,0.000003,...,0.000027,0.000002,3.813068e-07,6.355113e-07,1.016818e-06,0.000004,3.086769e-07,9.623457e-07,0.002555,0.002224
spinal_cord,3.119635e-07,5.361873e-07,0.000002,0.000015,0.000003,0.000036,0.000002,0.000004,0.000014,0.000003,...,0.000028,0.000002,7.116668e-07,5.166896e-07,8.578997e-07,0.000004,2.924658e-07,8.481508e-07,0.002512,0.002188
neural_optic2,3.404044e-07,1.565860e-06,0.000003,0.000015,0.000003,0.000032,0.000002,0.000005,0.000017,0.000003,...,0.000028,0.000002,3.857916e-07,5.900343e-07,8.169705e-07,0.000005,7.942769e-08,7.488897e-07,0.002488,0.002183


In [ ]:
#Aggregate by cell type and developmental stages
meta_ct_time = cell_types + "__" + dev_times #Build modified meta vector: cell type + developmental time

agg_atac_ct_time = agg_fun(atac_ct_matrix, meta_ct_time, fun="sum", peaks = peaks)
agg_atac_ct_time.head()


In [ ]:
# Checking the aggregation results by the shape of the resulting dataframe and its columns
print(agg_atac_ct.shape)
print(agg_atac_ct.index)
agg_atac_ct.columns.head()

print(agg_atac_ct_time.shape)
print(agg_atac_ct_time.index)
agg_atac_ct_time.columns.head()

(39, 640834)
Index(['epidermis', 'pronephros', 'hindbrain', 'spinal_cord', 'neural_optic2',
       'neural_floor_plate', 'neural_crest2', 'PSM', 'optic_cup',
       'lateral_plate_mesoderm', 'midbrain_hindbrain_boundary2',
       'neural_telencephalon', 'differentiating_neurons', 'muscle',
       'fast_muscle', 'heart_myocardium', 'somites', 'NMPs', 'epidermis2',
       'pharyngeal_arches', 'floor_plate2', 'hemangioblasts',
       'neural_posterior', 'floor_plate', 'tail_bud', 'endoderm',
       'midbrain_hindbrain_boundary', 'neural_crest', 'neural_optic',
       'hematopoietic_vasculature', 'endocrine_pancreas', 'hatching_gland',
       'neurons', 'notochord', 'pronephros2', 'enteric_neurons', 'epidermis3',
       'epidermis4', 'neural'],
      dtype='object')


Index(['1-32-526', '1-2372-3057', '1-3427-4032', '1-4469-7268', '1-9541-9969',
       '1-11007-12962', '1-13276-13705', '1-14059-14260', '1-14625-15105',
       '1-15724-15934',
       ...
       '25-37489449-37490741', '25-37492035-37493157', '25-37493175-37493740',
       '25-37496420-37496948', '25-37497049-37497789', '25-37498106-37500090',
       '25-37500598-37500859', '25-37501104-37501839', 'MT-22-3567',
       'MT-13233-16532'],
      dtype='object', length=640834)

In [ ]:
#Comparing the number of combinations of cell types and developmental stages ("meta_ct_time") with the index (row) of the aggregated data frame to find out which groups are (not) present in the aggregated data frame
import numpy as np

meta_groups = set(meta_ct_time.unique())
agg_groups = set(agg_atac_ct_time.index)

print("Missing in ct + time dataframe:", meta_groups - agg_groups)
print("Number of combinations of cell types and developmental stages:", len(meta_groups))
print("Number of groups in the ct + time dataframe:", len(agg_groups))


In [7]:
#Subsetting some columns to get the mean values grouped by cell type for the metadata columns "nCount_RNA", "nCount_ATAC", "TSS.enrichment", "nucleosome_signal"
metadata_lg_ct = filtered_ATdata.obs[["nCount_RNA", "nCount_ATAC", "TSS.enrichment", "nucleosome_signal", "annotation_ML"]].copy()
metadata_lg_ct.head()

#group by cell type
metadata_lg_ct = metadata_lg_ct.groupby(["annotation_ML"]).mean().reset_index()
metadata_lg_ct.index = metadata_lg_ct["annotation_ML"].astype(str)

# Sort agg_atac_ct to ensure the same order of rows as metadata_lg_ct
agg_atac_ct = agg_atac_ct.reindex(metadata_lg_ct.index)

In [31]:
#Subsetting some columns to get the mean values grouped by cell type and developmental stage for the metadata columns "nCount_RNA", "nCount_ATAC", "TSS.enrichment", "nucleosome_signal"
metadata_lg = filtered_ATdata.obs[["nCount_RNA", "nCount_ATAC", "TSS.enrichment", "nucleosome_signal", "annotation_ML", "dev_stage"]].copy()

#group by cell type and developmental stage to get the mean values for the metadata columns "nCount_RNA", "nCount_ATAC", "TSS.enrichment", "nucleosome_signal"
metadata_lg_ct_time = metadata_lg.groupby(["annotation_ML", "dev_stage"], observed=True).mean().reset_index()
metadata_lg_ct_time.index = metadata_lg_ct_time["annotation_ML"].astype(str) + "__" + metadata_lg_ct_time["dev_stage"].astype(str)

# Sort agg_atac_ct_time by metadata_lg_ct_time index to ensure the same order of rows in both data frames
agg_atac_ct_time = agg_atac_ct_time.reindex(metadata_lg_ct_time.index)

NameError: name 'agg_atac_ct_time' is not defined

In [8]:
# Save the aggregated data frames to a new AnnData object
import pandas as pd
agg_atac_ct = AnnData(X=agg_atac_ct.values, obs= metadata_lg_ct, var=filtered_ATdata.var)
agg_atac_ct.write_h5ad("/home/fgsasse_lrs_1/Downloads/BA/BA_data/Pseudobulks/ATAC/celltypes/agg_atac_ct.h5ad", compression="gzip")

#agg_atac_ct_time = AnnData(X=agg_atac_ct_time.values, obs=metadata_lg_ct_time, var=filtered_ATdata.var)
#agg_atac_ct_time.write_h5ad("/home/fgsasse_lrs_1/Downloads/BA/BA_data/Pseudobulks/ATAC/celltypes_times/agg_atac_ct_time.h5ad", compression="gzip")


In [ ]:
# Sanity chechk
timepoint = "0somites"
cell_type = "NMPs"
pks = "1-32-526"
mask = (filtered_ATdata.obs["dev_stage"]==timepoint) & (filtered_ATdata.obs["annotation_ML"]==cell_type)

# subset the X matrix for the selected cells and peaks
original_counts = filtered_ATdata.X[mask.values, filtered_ATdata.var_names.get_loc(pks)]
original_sum_counts = filtered_ATdata.X[mask.values, :].sum()
target_value = original_counts / original_sum_counts
print(target_value)

calculated_value = agg_atac_ct_time_anndata[cell_type + "__" + timepoint, agg_atac_ct_time_anndata.var_names.get_loc(pks)]
print(calculated_value)